# Milestone 1: Data Collection, Preprocessing & Baseline CRNN

This notebook covers:
1. Setup and dataset download
2. Dataset exploration and statistics
3. Image preprocessing pipeline
4. Baseline CRNN model training
5. Results and error analysis

## 1. Setup

In [ ]:
# Run on Colab: clone repo and install dependencies
import os
if 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ:
    !git clone https://github.com/<YOUR_USERNAME>/Image-processing-project.git
    %cd Image-processing-project
    !pip install -q -r requirements.txt
else:
    # Running locally — make sure we're in the project root
    %cd ..

import sys
sys.path.insert(0, '.')

## 2. Load IAM Dataset (via HuggingFace)

No manual download needed — the dataset loads automatically from HuggingFace Hub.

In [ ]:
from src.dataset import load_iam_splits, IAMDataset, LabelEncoder

print('Loading IAM dataset from HuggingFace (first run downloads ~1GB)...')
train_hf, val_hf, test_hf = load_iam_splits()

print(f'Train: {len(train_hf)} samples')
print(f'Val:   {len(val_hf)} samples')
print(f'Test:  {len(test_hf)} samples')

## 3. Dataset Exploration

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

# Label length distribution
label_lengths = [len(sample['text']) for sample in train_hf]
print(f'Avg label length: {np.mean(label_lengths):.1f}')
print(f'Max label length: {max(label_lengths)}')

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.hist(label_lengths, bins=30, edgecolor='black')
plt.xlabel('Text Length (chars)')
plt.ylabel('Count')
plt.title('Label Length Distribution')

# Character frequency
all_chars = Counter()
for sample in train_hf:
    all_chars.update(sample['text'].lower())

chars, counts = zip(*all_chars.most_common(30))
plt.subplot(1, 2, 2)
plt.bar(range(len(chars)), counts)
plt.xticks(range(len(chars)), chars)
plt.xlabel('Character')
plt.ylabel('Frequency')
plt.title('Top 30 Character Frequencies')
plt.tight_layout()
plt.show()

In [ ]:
# Show sample images
fig, axes = plt.subplots(4, 5, figsize=(20, 8))
indices = np.random.choice(len(train_hf), 20, replace=False)

for ax, idx in zip(axes.flat, indices):
    sample = train_hf[int(idx)]
    ax.imshow(sample['image'], cmap='gray')
    ax.set_title(f'"{sample["text"]}"', fontsize=10)
    ax.axis('off')

plt.suptitle('Sample Images from IAM Dataset', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Preprocessing Pipeline Demo

In [ ]:
from src.preprocessing import build_preprocessing_pipeline
import cv2

transform = build_preprocessing_pipeline(img_height=32, img_width=128)

# Show preprocessing on 5 samples
indices = np.random.choice(len(train_hf), 5, replace=False)

for idx in indices:
    sample = train_hf[int(idx)]
    img = sample['image']
    if img.mode != 'L':
        img = img.convert('L')
    img_np = np.array(img, dtype=np.uint8)

    # Show original vs preprocessed
    processed = transform(img_np)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 2))
    ax1.imshow(img_np, cmap='gray')
    ax1.set_title(f'Original: "{sample["text"]}"')
    ax1.axis('off')

    ax2.imshow(processed.squeeze().numpy(), cmap='gray')
    ax2.set_title('Preprocessed')
    ax2.axis('off')
    plt.tight_layout()
    plt.show()

## 5. Build Baseline CRNN Model

In [ ]:
import torch
from src.model import CRNN
from src.dataset import LabelEncoder

encoder = LabelEncoder()
model = CRNN(num_classes=len(encoder))

total_params = sum(p.numel() for p in model.parameters())
print(f'Number of classes: {len(encoder)}')
print(f'Total parameters: {total_params:,}')
print()
print(model)

# Verify with dummy input
dummy = torch.randn(2, 1, 32, 128)
out = model(dummy)
print(f'\nInput shape:  {dummy.shape}')
print(f'Output shape: {out.shape}  (T={out.shape[0]}, B={out.shape[1]}, C={out.shape[2]})')

## 6. Train Baseline Model

In [ ]:
from src.train import train

results = train('configs/baseline_crnn.yaml', checkpoint_dir='checkpoints')

## 7. Results & Sample Predictions

In [ ]:
from src.utils import show_predictions, load_checkpoint, plot_training_curves
import yaml

# Reload best model
with open('configs/baseline_crnn.yaml', 'r') as f:
    config = yaml.safe_load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
encoder = LabelEncoder()
model = CRNN(num_classes=len(encoder)).to(device)
checkpoint = load_checkpoint('checkpoints/best_model.pt', model)
print(f"Best model from epoch {checkpoint['epoch'] + 1}")
print(f"Best CER: {checkpoint['metrics']['cer']:.4f}")
print(f"Best WER: {checkpoint['metrics']['wer']:.4f}")

# Plot training curves
if results:
    plot_training_curves(results['train_losses'], results['val_losses'], results['val_cers'])

In [ ]:
from src.dataset import get_dataloaders

_, val_loader, _ = get_dataloaders(config, encoder)

print('=== Sample Predictions ===')
show_predictions(model, val_loader, encoder, device, n=15)

## 8. Error Analysis

In [ ]:
from src.metrics import compute_cer, evaluate_batch

# Collect all predictions and find worst ones
model.eval()
all_predictions = []

with torch.no_grad():
    for images, labels, label_lengths, input_lengths in val_loader:
        images = images.to(device)
        log_probs = model(images)
        results_eval = evaluate_batch(log_probs.cpu(), labels, label_lengths, encoder)
        for pred, target in zip(results_eval['predictions'], results_eval['targets']):
            cer = compute_cer(pred, target)
            all_predictions.append((cer, pred, target))

all_predictions.sort(key=lambda x: x[0], reverse=True)

print('=== Worst 20 Predictions (highest CER) ===')
print(f'{"CER":>6} | {"Predicted":>20} | {"Ground Truth":>20}')
print('-' * 55)
for cer, pred, target in all_predictions[:20]:
    print(f'{cer:6.2f} | {pred:>20} | {target:>20}')

print(f'\n=== Overall Statistics ===')
cers = [x[0] for x in all_predictions]
print(f'Mean CER: {np.mean(cers):.4f}')
print(f'Median CER: {np.median(cers):.4f}')
print(f'Samples with CER=0 (perfect): {sum(1 for c in cers if c == 0)} / {len(cers)}')